# EDA: credit card fraud
All logic lives in `fraud.data` and `fraud.eda`. Overview plots use the deduplicated data; feature separation and correlations use the **train split only**, so nothing about val/test informs later choices. `V1`-`V28` are anonymised PCA components, so no business meaning is attached to them.

In [1]:
import logging

from fraud import data, eda

logging.basicConfig(level=logging.INFO, format="%(message)s")
raw = data.load_raw()
print(data.validate_schema(raw))
df, removed = data.dedupe(raw)
splits = data.time_split(df)
train = splits["train"]

Schema report: {'rows': 284807, 'nulls': 0, 'duplicates': 1081, 'non_numeric': [], 'frauds': 492}


{'rows': 284807, 'nulls': 0, 'duplicates': 1081, 'non_numeric': [], 'frauds': 492}


Removed 1081 duplicate rows (284807 -> 283726)


train: 198608 rows, 366 frauds


val: 42559 rows, 55 frauds


test: 42559 rows, 52 frauds


## Class imbalance

In [2]:
eda.plot_class_balance(df)

Saved D:\agentic AI\FraudLens\reports\figures\class_balance.png


{'legit': 283253, 'fraud': 473, 'fraud_rate': np.float64(0.001667101358352777)}

## Amount and Time by class

In [3]:
eda.plot_amount_time(df)

Saved D:\agentic AI\FraudLens\reports\figures\amount_time_by_class.png


,count,mean,std,min,25%,50%,75%,max
Class,,,,,,,,
0,283253.0,88.413575,250.379023,0.0,5.67,22.00,77.46,25691.16
1,473.0,123.871860,260.211041,0.0,1.00,9.82,105.89,2125.87


## Feature separation (train split, top 8 by |corr| with Class)

In [4]:
eda.plot_top_features(train, k=8)

Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


...

Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Saved D:\agentic AI\FraudLens\reports\figures\top_features_by_class.png


['V17', 'V14', 'V12', 'V10', 'V3', 'V16', 'V7', 'V11']

## Correlation structure (train split)

In [5]:
eda.plot_correlation(train).head(10)

Saved D:\agentic AI\FraudLens\reports\figures\correlation_heatmap.png


V17    0.368548
V14    0.314582
V12    0.265490
V10    0.243456
V3     0.230077
V16    0.222214
V7     0.218576
V11    0.163748
V4     0.141771
V18    0.132632
Name: Class, dtype: float64

## Fraud rate by hour
Time is seconds since the first transaction, so the hour is relative, not clock time.

In [6]:
eda.plot_fraud_by_hour(df)

Saved D:\agentic AI\FraudLens\reports\figures\fraud_rate_by_hour.png


,rate,frauds,n
Time,,,
0,0.000785,6,7647
1,0.002376,10,4208
2,0.014510,48,3308
3,0.004875,17,3487
4,0.010436,23,2204
5,0.003681,11,2988
6,0.002205,9,4082
7,0.003180,23,7233
8,0.000880,9,10232


## Takeaways
- **Extreme imbalance.** After dedup there are 473 frauds in 283,726 rows (0.167%), so accuracy is meaningless and PR-AUC is the headline metric.
- **Amount alone does not separate classes.** Fraud median is 9.82 vs 22.00 for legit, but the means are 123.87 vs 88.41 and both have long tails. Fraud has many tiny amounts (25th percentile 1.00).
- **A handful of components carry most of the signal.** On the train split the largest |corr| with Class are V17 (0.37), V14 (0.32), V12 (0.27), V10 (0.24) and V3 (0.23). These are anonymised PCA components, so no business meaning is claimed.
- **Features are nearly decorrelated.** The largest pairwise |corr| among V1-V28 on train is 0.10, as expected from PCA, so multicollinearity is not a concern for linear models.
- **Fraud rate is time-varying.** Hours 2 and 4 (relative to the first transaction, folded over the two days) show about 1.5% and 1.0% fraud vs roughly 0.1% in most hours, on small volumes (2-3k rows). This is a 2-day window, so it is suggestive only; the clock offset is unknown.
- **Implication for evaluation.** Fraud patterns shift over time and val/test hold only 55 and 52 frauds, so a chronological split will score below published random-split numbers and results need bootstrap CIs.